In [ ]:
!pip install -U stanza
import stanza as sz
from stanza.models.coref.model import Config

sz.download('en')

# set plateau epochs variable in the init function
old_init = Config.__init__
def new_init(self, *args, **kwargs):
  if 'plateau_epochs' not in kwargs:
    kwargs['plateau_epochs'] = None
  old_init(self, *args, **kwargs)

Config.__init__ = new_init

nlp = sz.Pipeline('en',processors="tokenize,mwt,pos,lemma,depparse,ner,coref")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 41.3 MB/s eta 0:00:00


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json


INFO:stanza:Loading these models for language: en (English):
| Processor | Package                   |
-----------------------------------------
| tokenize  | combined                  |
| mwt       | combined                  |
| pos       | combined_charlm           |
| lemma     | combined_nocharlm         |
| coref     | udcoref_xlm-roberta-lora  |
| depparse  | combined_charlm           |
| ner       | ontonotes-ww-multi_charlm |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: coref
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebook

...

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


In [ ]:
import pandas as pd
df = pd.read_csv("drive/MyDrive/MWP_data/mawps-dev.csv")
txts = []
for v in df['source']:
  txts.append(v)
#txts now has each of the individual sentences
docs = [ nlp(x) for x in txts] #do nlp

In [ ]:
docs[0].text

'In a bag there are 13 red marbles, 5 blue marbles, and 7 green marbles.'

In [ ]:
'''
def getCoref(sentence):
  corefs = {}
  for word in sentence.words:
    for attatchment in word.coref_chains:
      i = attatchment.chain.index
      if not i in corefs.keys():
        corefs[i] = attatchment.chain
  return list(corefs.values())

def getCorefs(sentences):
  if type(sentences) == list:
    return [getCoref(x) for x in sentences]
  else:
    return getCoref(sentences)

[[cr.representative_text for cr in l] for l in getCorefs(docs[-2].sentences)]
'''

[['Her', 'Her dad', '5 more nickels']]

In [ ]:
'''
# Create a list of entities and quantities for a sentences
def parseDoc(doc):
  words = [word for sentence in doc.sentences for word in sentence.words]
  tokens = [token for sentence in doc.sentences for token in sentence.tokens]
  # non quantity things
  entities = []
  # quantity things
  quantities = []
  q_indices = []

  #First find quantity things by looking at the ents
  for i,w in enumerate(words):
    #ignore words with more than 1 or 0 coreferences
    if len(w.coref_chains) != 1:
      continue
    ref = w.coref_chains[0].chain
    #if the token is a number
    if tokens[i].ner in ["S-CARDINAL","B-QUANTITY"]:
      #add the representative text of the number to quantities
      quantities.append(ref.representative_text)
      q_indices.append(ref.index)
    # if not a number, and not part of the representative text of a number, assume entity
    elif ref.index not in q_indices:
      entities.append(ref.representative_text)
      q_indices.append(ref.index)
    else:
      # all other representative texts are entities
      for n in range(1, len(w.coref_chains)):
        ref =  w.coref_chains[n].chain
        if ref.index not in q_indices:
          entities.append(ref.representative_text)
          q_indices.append(ref.index)


  return entities , quantities

entities,quantities = parseDoc(docs[-2])
print("Entities")
for l in entities:
  print(l)
print("\nQuantities")
for e in quantities:
  print(e)
'''

Entities
Her dad
Her

Quantities
5 more nickels


In [ ]:
import re

class Quantity:
  def __init__(self, obj):
    if "entity" in obj.keys():
      self.entity = obj["entity"]
    else:
      self.entity = None

    if "quantity" in obj.keys():
      self.quantity = obj["quantity"]
      self.has_quantity = True
    else:
      self.has_quantity = False

    if "attrs" in obj.keys():
      self.attrs = obj["attrs"]

    if "unit" in obj.keys():
      self.unit = obj["unit"]
      self.has_unit = True
    else:
      self.has_unit = False

    if "verb" in obj.keys():
      self.verb = obj["verb"]
      self.has_verb = True
    else:
      self.has_verb = False

    if "corefIndex" in obj.keys():
      self.corefIndex = obj["corefIndex"]
      self.has_coref = True
    else:
      self.has_coref = False

    if "startIndex" in obj.keys():
      self.startIndex = obj["startIndex"]

    if "endIndex" in obj.keys():
      self.endIndex = obj["endIndex"]

    #TODO add text_spans list of tuples (start end) of text in the doc to repalce with this quant

  def __str__(self):
    s = ""
    if self.has_verb:
      s += f"{self.verb} "
    if self.has_quantity:
      s += f"{self.quantity} "
    else:
      s += "x "
    if self.has_unit:
      s += f"{self.unit} of "
    for a in self.attrs:
      s += a + " "

    s += self.entity if self.entity else ""
    return s

# traverses the word tree to find all words that are children of some word
def getWordChildIds(tupleList, word):
  children = []
  childrenUpdate = True
  while childrenUpdate:
    childrenUpdate = False
    for tup in tupleList:
      if(tup[0].head in children or tup[0].head == word.id) and tup[0].id not in children:
        childrenUpdate = True
        children.append(tup[0].id)
  return children

#Get structured quantities from the document
def parseQuantities(sentence):
  words = [word for word in sentence.words]
  tokens = [token for token in sentence.tokens]
  quantities = []
  referencedChains = []
  for i,w in enumerate(words):
    token = next((t for t in tokens if w.id in t.id),None)

    if not token:
      #skip this word if no token represents it
      continue

    if w.pos == "NUM" or token.ner in ["S-CARDINAL","B-QUANTITY","S-MONEY","B-MONEY", "B-TIME", "I-QUANTITY"]:
      #print("CHAIN LENGTH", len(w.coref_chains))
      #for chain in w.coref_chains:
        #print("->",chain.chain.representative_text)
      if len(w.coref_chains) < 1:
        quant = [(w,token)]
        NUMParentId = w.head
        NUMParent = next((word for word in words if (word.id == NUMParentId)),None)
        if NUMParentId > w.id:
          parentToken = next((t for t in tokens if NUMParentId in t.id),None)
          quant.append((NUMParent,parentToken))
        quantities.append(quant)
        #comment out the line below V
        #quantities.append([(w,token)])
      else:
        #create list of all adjacent words in the coref
        #use the shortest coreference to avoid masking unnecessarily
        shortestCoref = w.coref_chains[0].chain
        for c in w.coref_chains[1:]:
          if len(c.chain.representative_text) < len(shortestCoref.representative_text):
            shortestCoref = c.chain
        coref_index = shortestCoref.index
        if coref_index in referencedChains:
          #do not create a quantity for the same coreference twice
          continue
        referencedChains.append(coref_index)
        quant = []
        #prepend all words before the number that are part of the reference
        word_index = i
        while word_index >= 0 and len(words[word_index].coref_chains)>0 and coref_index in [c.chain.index for c in words[word_index].coref_chains]:
          quant.insert(0,(words[word_index],token))
          word_index -= 1
        #append all words after the number that are part of the reference
        word_index  = i + 1
        while word_index < len(words) and len(words[word_index].coref_chains)>0 and coref_index in [c.chain.index for c in words[word_index].coref_chains]:
          quant.append((words[word_index],token))
          word_index += 1

        quantities.append(quant)

  # For each quantity create a labeled object
  res = []
  for quant in quantities:
    q = {}
    num = next((w for w in quant if (w[0].upos == "NUM" or w[0].deprel == "nummod")), None)
    if not num:
      #TODO handle some unknown quantities here -> may be unnecessary
      continue
    q["quantity"] = num[0].lemma
    if len(num[0].coref_chains) > 0:
      q["corefIndex"] = num[0].coref_chains[0].chain.index
    q["startIndex"] = quant[0][0].start_char
    q["endIndex"] = quant[-1][0].end_char
    #test what is in the quant
    #print([z[0].text for z in quant])

    # find if there is a unit describing the number or an entity the number quantifies
    #does num have a parent in quant?
    numParentId = num[0].head
    parentOfNum = next((w for w in quant if (w[0].id == numParentId)), None)
    if parentOfNum:
      parentOfNum = parentOfNum[0] #just the word, not the token
    if parentOfNum:
      #find if the parent of num has any children with deprel nmod
      #find ids of all the parent's children
      children = getWordChildIds(quant, parentOfNum)
      children.remove(num[0].id)
      # check if any of these children are nmod
      possibleEnt = None
      for tup in quant:
        #possible entities are children who modify the quantity (nmod) not including pronouns
        if tup[0].id in children and tup[0].deprel == "nmod" and (tup[0].pos != "PRON"):
          possibleEnt = tup
          break
      if possibleEnt:
        #parent is only automatic unit if in quant, not just words
        q["unit"] = parentOfNum.lemma
        entStr = possibleEnt[0].lemma
        # gather compoun word entities like "ice cream"
        for tup in quant:
          if tup[0].head == possibleEnt[0].id and tup[0].deprel == "compound":
            entStr = tup[0].lemma + " " + entStr
        q["entity"] = entStr
      elif parentOfNum.pos not in ["ADJ"]:
        #no children are nmod - parentOfNum is the entity (unless it is an ADJ)
        entStr = parentOfNum.lemma
        # gather compoun word entities like "ice cream"
        for tup in quant:
          if tup[0].head == parentOfNum.id and tup[0].deprel == "compound":
            entStr = tup[0].lemma + " " + entStr
        q["entity"] = entStr
    else:
      #num has no parent - find a child of num in quant that is nmod to be the entity
      children = getWordChildIds(quant, num[0])
      possibleEnt = None
      for tup in quant:
        #possible entities are children who modify the quantity (nmod) not including pronouns
        if tup[0].id in children and tup[0].deprel == "nmod" and (tup[0].pos != "PRON"):
          possibleEnt = tup
          break
      if possibleEnt:
        entStr = possibleEnt[0].lemma
        for tup in quant:
          if tup[0].head == possibleEnt[0].id and tup[0].deprel == "compound":
            entStr = tup[0].lemma + " " + entStr
        q["entity"] = entStr
      #else case here is only path where you have no entity associated with the quantity

    #collect any attributes
    attrs = []
    for tup in quant:
      if tup[0].deprel == "amod":
        attrs.append(tup[0].lemma)

    # include the sentence's root ver
    # TODO, make a special attribute in the class for the verb
    possibleVerb = num[0]
    while possibleVerb.pos != "VERB" and possibleVerb.head > 0:
      possibleVerb = next((w for w in words if w.id == possibleVerb.head),None)
    if possibleVerb.pos == "VERB":
      q["verb"] = possibleVerb.lemma
    q["attrs"] = attrs

    qClass = Quantity(q)
    res.append(qClass)

  # end for loop
  return res

def replaceQuantities(doc, logging=False):
  quantities = []
  for sentence in doc.sentences:
    quantities += parseQuantities(sentence)
  text = doc.text
  newText = ""
  entities = []
  units = []
  attrs = []
  # add all unique things to their respective lists
  for q in quantities:
    if q.entity and q.entity not in entities:
      #if this entity is already a unit, make this q's unit the entity
      #and it's entity the same as q that originally introduced the unit
      if q.entity in units and not q.has_unit:
        q.unit = q.entity
        q.has_unit = True
        entMirror = next((quant for quant in quantities if quant.has_unit and quant.unit == q.unit),None)
        q.entity = entMirror.entity
      else:
        entities.append(q.entity)
    if q.has_unit and q.unit not in units:
      units.append(q.unit)
    for attr in q.attrs:
      if attr not in attrs:
        attrs.append(attr)
  # replace coreferences in the sentence with generic tokens
  for i,q in enumerate(quantities):
    #represent generic quantity
    qStr = f"[Q{i}] "
    #represent generic unit if applicable
    if q.has_unit:
      unitIndex = units.index(q.unit)
      qStr += f"[U{unitIndex}] "
    for attr in q.attrs:
      attrIndex = attrs.index(attr)
      qStr += f"[A{attrIndex}] "
    if q.entity:
      entIndex = entities.index(q.entity)
      qStr += f"[E{entIndex}]"
    else:
      #if an earlier quantity shares the coreference use that
      corefQuantity = None
      if q.has_coref:
        corefQuantity = next((c for c in quantities[:i] if c.has_coref and c.corefIndex == q.corefIndex),None)
      if corefQuantity:
        entIndex = entities.index(corefQuantity.entity)
      else:
        #assume the entity used by previous quantity
        j = i
        while j >= 0 and not quantities[j].entity:
          j -= 1
        if quantities[j].entity:
          entIndex = entities.index(quantities[j].entity)
        else:
          #assume first entity referenced if no previous quantity has one
          entIndex = 0
        #print("substituting entity:", entIndex, entities, "with quantity:", q.quantity)
      qStr += f"[E{entIndex}]"

    #print("qStr for quantity", q.quantity, "is", qStr)
    #combine qStr and the text
    if i == 0:
      #start
      newText += text[:q.startIndex] + qStr
    elif i == len(quantities) - 1:
      #end
      newText += text[quantities[i-1].endIndex:q.startIndex] + qStr + text[q.endIndex:]
    else:
      #middle
      newText += text[quantities[i-1].endIndex:q.startIndex] + qStr


    #generate a key for the generics
    genToOriginal = {}
    originalToGen = {}

    for i,e in enumerate(entities):
      genToOriginal[f"[E{i}]"] = e
      originalToGen[e] = f"[E{i}]"
    for i,e in enumerate(units):
      genToOriginal[f"[U{i}]"] = e
      originalToGen[e] = f"[U{i}]"
    for i,e in enumerate(attrs):
      genToOriginal[f"[A{i}]"] = e
      originalToGen[e] = f"[A{i}]"
    for i,q in enumerate(quantities):
      genToOriginal[f"[Q{i}]"] = q.quantity
      originalToGen[q.quantity] = f"[Q{i}]"

    # replace non-quantity versions of genericized words
    for sent in doc.sentences:
      for word in sent.words:
        if word.lemma in originalToGen.keys():
          #replace this word in newText with its corresponding generic
          newText = newText.replace(word.text, originalToGen[word.lemma])
    #catch compound lemmas
    for k in originalToGen.keys():
      newText = newText.replace(k,originalToGen[k])

  #replace non-quantity entities in the text
  singleCorefs = []
  for sent in doc.sentences:
      for word in sent.words:
        if len(word.coref_chains) == 1:
          singleCorefs.append(word.coref_chains[0])

  if logging: print("sC", [s.chain.representative_text for s in singleCorefs])

  entityRepTexts = []

  for coref in singleCorefs:
    if coref.chain.representative_text in newText:

      if logging: print("\nTEXT APPEARS", coref.chain.representative_text,"\n")

      blocks = []
      newCorefIndex = len(entityRepTexts)
      entityRepTexts.append(coref.chain.representative_text)
      for sent in doc.sentences:
        words = sent.words

        if logging: print([w.text for w in words])

        for i in range(len(words)):

          if logging: print(i, words[i].text, len(words), len(words[i].coref_chains))

          wordCorefs = [c.chain.index for c in words[i].coref_chains]
          block = []
          while i<len(words) and coref.chain.index in wordCorefs:
            block.append(words[i].text)
            i += 1
            wordCorefs = [c.chain.index for c in words[i].coref_chains]
          if len(block) > 0:
            blocks.append(block)
      #blocks now has lines that can be replaced by the coref values
      for b in blocks:
        target_text = re.escape(" ".join(b))
        pattern = rf"(?<![a-zA-Z]){target_text}(?![a-zA-Z])"
        newText = re.sub(pattern, f"[P{newCorefIndex}]", newText)
        #old
        #newText = newText.replace(" ".join(b), f"[P{newCorefIndex}]")

  for i,e in enumerate(entityRepTexts):
      genToOriginal[f"[P{i}]"] = e
      originalToGen[e] = f"[P{i}]"

  return newText, genToOriginal, originalToGen


# Testin parse Quantities
def testParsing ():
  for i,doc in enumerate(docs):
    parse = []
    for sentence in doc.sentences:
      parse.append(parseQuantities(sentence))
    print(f"{i} Original:")
    for s in doc.sentences:
      print(s.text)
    #print("------------------------")
    # display the found quantities
    #for parsedSent in parse:
    #  for p in parsedSent:
    #    cI = None
    #    if p.has_coref:
    #      cI = p.corefIndex
    #    print(p,"|", cI, p.startIndex, p.endIndex)
    print("------------------------")
    newText, gTO, oTG = replaceQuantities(doc)
    print(gTO,"\n",oTG)
    print("------------------------")
    print(newText)
    print("\n ~~~~~~~~~ \n")

#testParsing()


#specific test
#print(replaceQuantities(docs[60]))


In [ ]:
import difflib
from tqdm.auto import tqdm
tqdm.pandas() #patches in some functions

train_df = pd.read_csv("drive/MyDrive/MWP_data/mawps-train.csv")
train_df = train_df.drop(columns=['prev_source', 'prev_target'])

#so empty targets can be joined

train_df['target'] = train_df['target'].fillna('')

train_df = train_df.groupby('problem_id').agg({
    'source': ' '.join,
    'target': lambda x: ' '.join(x.astype(str))
}).reset_index()

#TODO comment out below to use the whole df
#train_df = train_df.head()

def maskSource(source):
  #source is the original text
  doc = nlp(source)
  print("masking",doc.text)
  newText, gTO, oTG = replaceQuantities(doc, logging=True)
  return newText, gTO, oTG

x,y,z = maskSource(train_df['source'][43])

#train_df[['maskedSource','oTG_key', 'gTO_key']] = train_df['source'].progress_apply(
#    lambda x: pd.Series(maskSource(x))
#)




masking A spaceship traveled 0.5 light-year from Earth to Planet X and 0.1 light-year from Planet X to Planet Y.. Then it traveled 0.1 light-year from Planet Y back to Earth. How many light-years did the spaceship travel in all?
sC ['A spaceship', 'A spaceship', '0.5 light - year', '0.5 light - year', '0.5 light - year', '0.5 light - year', 'Earth', 'Planet X', 'Planet X', '0.1 light - year', '0.1 light - year', '0.1 light - year', '0.1 light - year', 'Planet X', 'Planet X', 'Planet Y', 'Planet Y', 'A spaceship', '0.1 light - year', '0.1 light - year', '0.1 light - year', '0.1 light - year', 'Planet Y', 'Planet Y', 'Earth', 'How many light - years', 'How many light - years', 'How many light - years', 'How many light - years', 'How many light - years', 'A spaceship', 'A spaceship']

TEXT APPEARS A spaceship 

['A', 'spaceship', 'traveled', '0.5', 'light', '-', 'year', 'from', 'Earth', 'to', 'Planet', 'X', 'and', '0.1', 'light', '-', 'year', 'from', 'Planet', 'X', 'to', 'Planet', 'Y..']


IndexError: list index out of range

In [ ]:
#use oTG key to mask the predicate
def maskPredicate(row):
  gTO = row['gTO_key']
  pred = row['target']

  #extract the meaningful words in predicate
  print(pred)
  groups = re.findall(r"\((.*?)\)", pred)

  words = []
  for g in groups:
      res = re.findall(r"\b\w+\b", g)
      words += res

  words = [w for w in words if w != "None"]
  wordSet = set(words)

  maskedPred = pred
  choices = list(gTO.keys())
  for word in wordSet:
    #find closest key
    closestKey = difflib.get_close_matches(word, choices, n=1, cutoff=0)
    if len(closestKey) > 0:
      pattern = rf"(?<![a-zA-Z]){word}(?![a-zA-Z])"
      maskedPred = re.sub(pattern, gTO[closestKey[0]], maskedPred)
      #maskedPred = maskedPred.replace(word, gTO[closestKey[0]])

  print(maskedPred)
  return maskedPred

train_df['maskedPredicate'] = train_df.progress_apply(maskPredicate, axis=1)

train_df

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-distilroberta-v1')

#use util.cos_sim(e1, e2) to compare cosine similarity



In [ ]:
train_df['maskedPredEmbedding'] = train_df['maskedSource'].progress_apply(lambda x: model.encode(x))
train_df.to_csv("drive/MyDrive/MWP_data/mawps-cb.csv")
train_df